# Olist Preprocessing Pipeline

This notebook implements a full preprocessing pipeline for the Brazilian E-Commerce Public Dataset by Olist, designed for downstream joins with Finnhub daily stock features.

In [ ]:
from pathlib import Path

import numpy as np

import pandas as pd



pd.set_option('display.max_columns', 200)

pd.set_option('display.width', 200)



PROJECT_ROOT = Path().resolve().parent

INPUT_DIR = PROJECT_ROOT / 'archive'

OUTPUT_DIR = PROJECT_ROOT / 'processed'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



REQUIRED_FILES = {

    'orders': 'olist_orders_dataset.csv',

    'order_items': 'olist_order_items_dataset.csv',

    'payments': 'olist_order_payments_dataset.csv',

    'reviews': 'olist_order_reviews_dataset.csv',

    'customers': 'olist_customers_dataset.csv',

    'sellers': 'olist_sellers_dataset.csv',

    'products': 'olist_products_dataset.csv',

    'geolocation': 'olist_geolocation_dataset.csv',

}



quality_issues = []

## 1) Load and Inspect

Load all 8 CSVs, inspect shape/dtypes/nulls/duplicates, and show key map.

In [6]:
def load_datasets(input_dir: Path):
    dfs = {}
    for name, fname in REQUIRED_FILES.items():
        fpath = input_dir / fname
        if not fpath.exists():
            raise FileNotFoundError(f'Required file not found: {fpath}')
        dfs[name] = pd.read_csv(fpath)
    return dfs

def inspect_dataframe(name: str, df: pd.DataFrame):
    print('=' * 90)
    print(f'[{name.upper()}]')
    print('=' * 90)
    print('Shape:', df.shape)
    print('\nDtypes:')
    print(df.dtypes)
    print('\nNull counts:')
    print(df.isna().sum().sort_values(ascending=False))
    print('\nDuplicate rows:', df.duplicated().sum())
    print()

dfs = load_datasets(INPUT_DIR)
for n, d in dfs.items():
    inspect_dataframe(n, d)

key_map = {
    'orders': {'primary': ['order_id'], 'foreign': ['customer_id']},
    'order_items': {'primary': ['order_id', 'order_item_id'], 'foreign': ['order_id', 'product_id', 'seller_id']},
    'payments': {'primary': ['order_id', 'payment_sequential'], 'foreign': ['order_id']},
    'reviews': {'primary': ['review_id'], 'foreign': ['order_id']},
    'customers': {'primary': ['customer_id'], 'foreign': ['customer_unique_id']},
    'sellers': {'primary': ['seller_id'], 'foreign': []},
    'products': {'primary': ['product_id'], 'foreign': []},
    'geolocation': {'primary': ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng'], 'foreign': []},
}

print('=' * 90)
print('PRIMARY/FOREIGN KEY MAP')
print('=' * 90)
for table, keys in key_map.items():
    print(f'{table}: primary={keys["primary"]}, foreign={keys["foreign"]}')

[ORDERS]
Shape: (99441, 8)

Dtypes:
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

Null counts:
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_estimated_delivery_date       0
dtype: int64

Duplicate rows: 0

[ORDER_ITEMS]
Shape: (112650, 7)

Dtypes:
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

Null 

## 2) Handle Missing Values

- Drop rows where `order_status`, `customer_id`, or `product_id` are null

- Impute `product_category_name` with `unknown`

- Impute `review_score` with median by product category (fallback: global median)

- Add boolean flags for imputation

In [7]:
orders = dfs['orders'].copy()
order_items = dfs['order_items'].copy()
payments = dfs['payments'].copy()
reviews = dfs['reviews'].copy()
customers = dfs['customers'].copy()
sellers = dfs['sellers'].copy()
products = dfs['products'].copy()
geolocation = dfs['geolocation'].copy()

before_orders = len(orders)
orders = orders.dropna(subset=['order_status', 'customer_id'])
if before_orders - len(orders) > 0:
    quality_issues.append(f'Dropped {before_orders - len(orders)} orders with null order_status/customer_id')

before_items = len(order_items)
order_items = order_items.dropna(subset=['product_id'])
if before_items - len(order_items) > 0:
    quality_issues.append(f'Dropped {before_items - len(order_items)} order_items rows with null product_id')

products['product_category_name_imputed'] = products['product_category_name'].isna()
products['product_category_name'] = products['product_category_name'].fillna('unknown')

def mode_or_nan(series: pd.Series):
    m = series.mode(dropna=True)
    return m.iloc[0] if len(m) else np.nan

review_category_bridge = (
    reviews[['review_id', 'order_id', 'review_score']]
    .merge(order_items[['order_id', 'product_id']], on='order_id', how='left')
    .merge(products[['product_id', 'product_category_name']], on='product_id', how='left')
)

global_median_review = reviews['review_score'].median()
category_medians = (
    review_category_bridge.groupby('product_category_name')['review_score']
    .median()
    .dropna()
    .to_dict()
)

review_score_map = (
    review_category_bridge.groupby('review_id')
    .agg(category_for_impute=('product_category_name', mode_or_nan), review_score=('review_score', 'first'))
    .reset_index()
)
review_score_map['category_median'] = review_score_map['category_for_impute'].map(category_medians)
review_score_map['review_score_imputed'] = review_score_map['review_score'].isna()
review_score_map['review_score_filled'] = np.where(
    review_score_map['review_score'].isna(),
    review_score_map['category_median'].fillna(global_median_review),
    review_score_map['review_score']
)

reviews = reviews.merge(
    review_score_map[['review_id', 'review_score_imputed', 'review_score_filled']],
    on='review_id',
    how='left'
)
reviews['review_score'] = reviews['review_score_filled'].astype(float)
reviews = reviews.drop(columns=['review_score_filled'])

imputed_reviews = int(reviews['review_score_imputed'].fillna(False).sum())
if imputed_reviews > 0:
    quality_issues.append(f'Imputed review_score for {imputed_reviews} rows using category/global medians')

## 3) Parse and Standardize Datetimes

Convert `*_date`, `*_timestamp`, and `*_at` columns to UTC datetimes and create `order_purchase_date` (daily).

In [8]:
def parse_datetimes_to_utc(df: pd.DataFrame):
    dt_cols = [c for c in df.columns if c.endswith('_date') or c.endswith('_timestamp') or c.endswith('_at')]
    for c in dt_cols:
        df[c] = pd.to_datetime(df[c], errors='coerce', utc=True)
    return df

orders = parse_datetimes_to_utc(orders)
order_items = parse_datetimes_to_utc(order_items)
payments = parse_datetimes_to_utc(payments)
reviews = parse_datetimes_to_utc(reviews)
customers = parse_datetimes_to_utc(customers)
sellers = parse_datetimes_to_utc(sellers)
products = parse_datetimes_to_utc(products)
geolocation = parse_datetimes_to_utc(geolocation)

orders['order_purchase_date'] = orders['order_purchase_timestamp'].dt.floor('D')

## 4) Remove Duplicates and Outliers

- Drop duplicate `order_id` in orders

- Cap `payment_value` and `freight_value` at 99th percentile

- Remove rows with `delivery_delay < -30` days

In [9]:
dup_order_ids = int(orders.duplicated(subset=['order_id']).sum())
if dup_order_ids > 0:
    quality_issues.append(f'Removed {dup_order_ids} duplicate order_id rows from orders')
orders = orders.drop_duplicates(subset=['order_id'], keep='first')

p99_payment = payments['payment_value'].quantile(0.99)
payment_outliers = int((payments['payment_value'] > p99_payment).sum())
payments['payment_value'] = payments['payment_value'].clip(upper=p99_payment)
if payment_outliers > 0:
    quality_issues.append(f'Capped {payment_outliers} payment_value rows at p99={p99_payment:.4f}')

p99_freight = order_items['freight_value'].quantile(0.99)
freight_outliers = int((order_items['freight_value'] > p99_freight).sum())
order_items['freight_value'] = order_items['freight_value'].clip(upper=p99_freight)
if freight_outliers > 0:
    quality_issues.append(f'Capped {freight_outliers} freight_value rows at p99={p99_freight:.4f}')

orders['_delivery_delay_days_raw'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.total_seconds() / 86400.0
bad_delay_mask = orders['_delivery_delay_days_raw'] < -30
bad_delay_count = int(bad_delay_mask.sum())
if bad_delay_count > 0:
    quality_issues.append(f'Removed {bad_delay_count} orders with delivery_delay_days < -30')
orders = orders.loc[~bad_delay_mask].copy()
orders = orders.drop(columns=['_delivery_delay_days_raw'])

## 5) Geolocation Aggregation

Aggregate geolocation by zip prefix and enrich customer/seller tables.

In [10]:
geo_agg = (
    geolocation.groupby('geolocation_zip_code_prefix', as_index=False)
    .agg(
        geolocation_lat=('geolocation_lat', 'mean'),
        geolocation_lng=('geolocation_lng', 'mean'),
        geolocation_state=('geolocation_state', mode_or_nan)
    )
    .rename(columns={'geolocation_zip_code_prefix': 'zip_code_prefix'})
)

customers['zip_code_prefix'] = customers['customer_zip_code_prefix']
customers = customers.merge(geo_agg, on='zip_code_prefix', how='left')
customers = customers.rename(columns={
    'geolocation_lat': 'customer_lat',
    'geolocation_lng': 'customer_lng',
    'geolocation_state': 'customer_geo_state'
})
customers['customer_state'] = customers['customer_state'].fillna(customers['customer_geo_state'])

sellers['zip_code_prefix'] = sellers['seller_zip_code_prefix']
sellers = sellers.merge(geo_agg, on='zip_code_prefix', how='left')
sellers = sellers.rename(columns={
    'geolocation_lat': 'seller_lat',
    'geolocation_lng': 'seller_lng',
    'geolocation_state': 'seller_geo_state'
})
sellers['seller_state'] = sellers['seller_state'].fillna(sellers['seller_geo_state'])

## 6) Merge Tables

Merge: orders -> order_items -> products -> payments -> reviews -> customers -> sellers using left joins and preserving all orders.

In [21]:
merged = (

    orders.merge(order_items, on='order_id', how='left', suffixes=('', '_order_item'))

    .merge(products, on='product_id', how='left', suffixes=('', '_product'))

    .merge(payments, on='order_id', how='left', suffixes=('', '_payment'))

    .merge(reviews, on='order_id', how='left', suffixes=('', '_review'))

    .merge(customers, on='customer_id', how='left', suffixes=('', '_customer'))

    .merge(sellers, on='seller_id', how='left', suffixes=('', '_seller'))

)



product_imputed = merged.get('product_category_name_imputed', pd.Series(False, index=merged.index)).infer_objects(copy=False)

product_imputed = product_imputed.astype('boolean').fillna(False).astype(bool)

review_imputed = merged.get('review_score_imputed', pd.Series(False, index=merged.index)).infer_objects(copy=False)

review_imputed = review_imputed.astype('boolean').fillna(False).astype(bool)



merged['has_imputed_values'] = product_imputed | review_imputed



print('Merged shape:', merged.shape)

Merged shape: (116147, 51)


## 7) Feature Engineering

Create all requested business and modeling features.

In [22]:
merged['delivery_delay_days'] = (merged['order_delivered_customer_date'] - merged['order_estimated_delivery_date']).dt.total_seconds() / 86400.0

merged['order_processing_time'] = (merged['order_approved_at'] - merged['order_purchase_timestamp']).dt.total_seconds() / 3600.0

merged['is_late_delivery'] = (merged['delivery_delay_days'] > 0).astype('Int64')



items_per_order = order_items.groupby('order_id')['order_item_id'].count().rename('items_per_order')

merged = merged.merge(items_per_order, on='order_id', how='left')



merged['payment_installments_ratio'] = merged['payment_installments'] / merged['items_per_order'].replace(0, np.nan)



sentiment_map = {1: 'negative', 2: 'negative', 3: 'neutral', 4: 'positive', 5: 'positive'}

merged['review_score_int'] = merged['review_score'].round().astype('Int64')

merged['review_sentiment'] = merged['review_score_int'].map(sentiment_map)



merged['product_volume_cm3'] = merged['product_length_cm'] * merged['product_height_cm'] * merged['product_width_cm']

## 8) Categorical Encoding

- Label encode `order_status`, `review_sentiment`, `payment_type`

- One-hot encode top 20 `product_category_name`, map rest to `other`

- Keep `customer_id` and `order_id` as identifiers

In [23]:
label_cols = ['order_status', 'review_sentiment', 'payment_type']

label_maps = {}



for col in label_cols:

    categories = sorted(merged[col].dropna().astype(str).unique())

    mapping = {cat: idx for idx, cat in enumerate(categories)}

    label_maps[col] = mapping

    merged[f'{col}_label'] = merged[col].astype(str).map(mapping)

    merged.loc[merged[col].isna(), f'{col}_label'] = np.nan



top20 = merged['product_category_name'].value_counts(dropna=False).head(20).index

merged['product_category_reduced'] = np.where(merged['product_category_name'].isin(top20), merged['product_category_name'], 'other')

cat_dummies = pd.get_dummies(merged['product_category_reduced'], prefix='product_cat', dtype=np.int8)

merged = pd.concat([merged, cat_dummies], axis=1)



# Portuguese -> English mapping for one-hot category columns

pt_to_en_category = {

    'cama_mesa_banho': 'bed_bath_table',

    'beleza_saude': 'health_beauty',

    'esporte_lazer': 'sports_leisure',

    'moveis_decoracao': 'furniture_decor',

    'informatica_acessorios': 'computers_accessories',

    'utilidades_domesticas': 'household_utilities',

    'relogios_presentes': 'watches_gifts',

    'telefonia': 'telephony',

    'ferramentas_jardim': 'garden_tools',

    'automotivo': 'automotive',

    'brinquedos': 'toys',

    'bebes': 'baby',

    'cool_stuff': 'cool_stuff',

    'eletronicos': 'electronics',

    'fashion_bolsas_e_acessorios': 'fashion_bags_accessories',

    'papelaria': 'stationery',

    'perfumaria': 'perfumery',

    'pet_shop': 'pet_shop',

    'moveis_escritorio': 'office_furniture',

}



rename_map = {}

for col in cat_dummies.columns:

    prefix = 'product_cat_'

    if col.startswith(prefix):

        suffix = col[len(prefix):]

        if suffix in pt_to_en_category:

            rename_map[col] = f"product_cat_{pt_to_en_category[suffix]}"



if rename_map:

    merged = merged.rename(columns=rename_map)

    cat_dummies = cat_dummies.rename(columns=rename_map)

## 9) Normalization

- Min-max: `payment_value`, `freight_value`, `product_weight_g`, `product_volume_cm3`

- Standard scale: `delivery_delay_days`, `order_processing_time`

- Do not scale binary flags or encoded categoricals

In [24]:
for raw_col in ['payment_value', 'freight_value']:
    merged[f'{raw_col}_raw'] = merged[raw_col]

minmax_cols = ['payment_value', 'freight_value', 'product_weight_g', 'product_volume_cm3']
for col in minmax_cols:
    cmin = merged[col].min(skipna=True)
    cmax = merged[col].max(skipna=True)
    if pd.isna(cmin) or pd.isna(cmax) or cmax == cmin:
        merged[f'{col}_minmax'] = 0.0
        quality_issues.append(f'Min-max scaling fallback for {col} due to degenerate range')
    else:
        merged[f'{col}_minmax'] = (merged[col] - cmin) / (cmax - cmin)

standard_cols = ['delivery_delay_days', 'order_processing_time']
for col in standard_cols:
    mean_v = merged[col].mean(skipna=True)
    std_v = merged[col].std(skipna=True, ddof=0)
    if pd.isna(std_v) or std_v == 0:
        merged[f'{col}_zscore'] = 0.0
        quality_issues.append(f'Standard scaling fallback for {col} due to zero std')
    else:
        merged[f'{col}_zscore'] = (merged[col] - mean_v) / std_v

## 10) Time-Series Alignment (Daily Features)

Aggregate to day level and enforce continuous date index with explicit missing-day handling.

In [25]:
daily = (

    merged.groupby('order_purchase_date', as_index=True)

    .agg(

        total_orders=('order_id', 'nunique'),

        total_revenue=('payment_value_raw', 'sum'),

        avg_review_score=('review_score', 'mean'),

        late_delivery_rate=('is_late_delivery', 'mean')

    )

    .sort_index()

)



if len(daily.index) > 0:

    full_range = pd.date_range(daily.index.min(), daily.index.max(), freq='D', tz='UTC')

    daily = daily.reindex(full_range)

    daily['total_orders'] = daily['total_orders'].fillna(0)

    daily['total_revenue'] = daily['total_revenue'].fillna(0)

    # Keep avg_review_score and late_delivery_rate as NaN on zero-order days.



daily.head()

,total_orders,total_revenue,avg_review_score,late_delivery_rate
2016-09-04 00:00:00+00:00,1.0,272.46,1.0,0.0
2016-09-05 00:00:00+00:00,1.0,75.06,1.0,0.0
2016-09-06 00:00:00+00:00,0.0,0.00,NaN,<NA>
2016-09-07 00:00:00+00:00,0.0,0.00,NaN,<NA>
2016-09-08 00:00:00+00:00,0.0,0.00,NaN,<NA>


## 11) Save Outputs + Final Summary

Write `olist_processed.parquet` and `daily_features.csv`, then print QA summary and quality flags.

In [26]:
merged_path = OUTPUT_DIR / 'olist_processed.parquet'
daily_path = OUTPUT_DIR / 'daily_features.csv'

merged.to_parquet(merged_path, index=False)
daily.to_csv(daily_path, index=True, index_label='date')

print('Saved merged dataset to:', merged_path)
print('Saved daily time-series to:', daily_path)

print('\n' + '=' * 90)
print('FINAL SUMMARY')
print('=' * 90)
print('Merged shape:', merged.shape)
if merged['order_purchase_date'].notna().any():
    print('Date range:', merged['order_purchase_date'].min(), 'to', merged['order_purchase_date'].max())
else:
    print('Date range: unavailable')
print('Total null count:', int(merged.isna().sum().sum()))
print('Feature count:', len(merged.columns))
print('Feature list:')
print(list(merged.columns))

print('\n' + '=' * 90)
print('DATA QUALITY FLAGS')
print('=' * 90)
if quality_issues:
    for issue in quality_issues:
        print('-', issue)
else:
    print('- No major data quality issues detected during preprocessing.')

Saved merged dataset to: processed\olist_processed.parquet
Saved daily time-series to: processed\daily_features.csv

FINAL SUMMARY
Merged shape: (116147, 92)
Date range: 2016-09-04 00:00:00+00:00 to 2018-10-17 00:00:00+00:00
Total null count: 221833
Feature count: 92
Feature list:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_purchase_date', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_imputed', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value', 'review_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer

In [17]:
# Post-run analysis snapshot for modeling and Finnhub alignment

summary = {}

summary['rows'] = int(len(merged))

summary['cols'] = int(merged.shape[1])

summary['unique_orders'] = int(merged['order_id'].nunique())

summary['unique_customers'] = int(merged['customer_id'].nunique())

summary['unique_products'] = int(merged['product_id'].nunique())

summary['unique_sellers'] = int(merged['seller_id'].nunique())

summary['date_min'] = str(merged['order_purchase_date'].min())

summary['date_max'] = str(merged['order_purchase_date'].max())

summary['overall_null_rate'] = float(merged.isna().sum().sum() / (merged.shape[0] * merged.shape[1]))

summary['imputed_row_rate'] = float(merged['has_imputed_values'].mean())

summary['late_delivery_rate'] = float(merged['is_late_delivery'].mean())

summary['median_delivery_delay_days'] = float(merged['delivery_delay_days'].median())

summary['p90_delivery_delay_days'] = float(merged['delivery_delay_days'].quantile(0.90))

summary['avg_processing_hours'] = float(merged['order_processing_time'].mean())

summary['median_payment_value_raw'] = float(merged['payment_value_raw'].median())

summary['total_revenue_raw'] = float(merged['payment_value_raw'].sum())



order_status_dist = merged[['order_id', 'order_status']].drop_duplicates()['order_status'].value_counts(normalize=True).round(4)

payment_type_dist = merged['payment_type'].value_counts(normalize=True).round(4)

review_sent_dist = merged['review_sentiment'].value_counts(normalize=True).round(4)

top_categories = merged['product_category_name'].value_counts(dropna=False).head(10)



daily_aligned = daily.copy()

daily_aligned.index = pd.to_datetime(daily_aligned.index, utc=True)

daily_corr = daily_aligned[['total_orders', 'total_revenue', 'avg_review_score', 'late_delivery_rate']].corr().round(3)



print('SUMMARY:')

print(summary)

print('\nORDER STATUS DISTRIBUTION:')

print(order_status_dist)

print('\nPAYMENT TYPE DISTRIBUTION:')

print(payment_type_dist)

print('\nREVIEW SENTIMENT DISTRIBUTION:')

print(review_sent_dist)

print('\nTOP 10 PRODUCT CATEGORIES:')

print(top_categories)

print('\nDAILY FEATURE CORRELATION:')

print(daily_corr)


SUMMARY:
{'rows': 116147, 'cols': 92, 'unique_orders': 97086, 'unique_customers': 97086, 'unique_products': 32256, 'unique_sellers': 3061, 'date_min': '2016-09-04 00:00:00+00:00', 'date_max': '2018-10-17 00:00:00+00:00', 'overall_null_rate': 0.02068302874056527, 'imputed_row_rate': 0.01449025803507624, 'late_delivery_rate': 0.07807347585387484, 'median_delivery_delay_days': -11.449652777777779, 'p90_delivery_delay_days': -1.1710648148148148, 'avg_processing_hours': 10.610460464733043, 'median_payment_value_raw': 107.78, 'total_revenue_raw': 18909705.02}

ORDER STATUS DISTRIBUTION:
order_status
delivered      0.9695
shipped        0.0114
canceled       0.0064
unavailable    0.0063
invoiced       0.0032
processing     0.0031
created        0.0001
approved       0.0000
Name: proportion, dtype: float64

PAYMENT TYPE DISTRIBUTION:
payment_type
credit_card    0.7361
boleto         0.1954
voucher        0.0543
debit_card     0.0143
not_defined    0.0000
Name: proportion, dtype: float64

REVIE

## 12) Project-Ready Artifacts for LLM + RAG

Create lightweight outputs that prepare this dataset for your assistant:

- Structured metadata profiles (schema + stats + sample values)

- Retrieval-friendly context chunks (JSONL)

- Business-friendly summary brief



> Note: This is preparation only (no vector store or LLM orchestration yet).

In [27]:
# Build RAG-ready metadata and chunk artifacts from processed outputs

import json



RAG_DIR = OUTPUT_DIR / 'rag'

RAG_DIR.mkdir(parents=True, exist_ok=True)



def profile_dataframe(df: pd.DataFrame, name: str, sample_n: int = 3):

    col_profiles = []

    for col in df.columns:

        s = df[col]

        non_null = s.dropna()

        profile = {

            'dataset': name,

            'column': col,

            'dtype': str(s.dtype),

            'row_count': int(len(s)),

            'null_count': int(s.isna().sum()),

            'null_rate': float(s.isna().mean()),

            'n_unique': int(non_null.nunique()) if len(non_null) else 0,

            'sample_values': [str(v) for v in non_null.head(sample_n).tolist()]

        }



        if pd.api.types.is_numeric_dtype(s):

            profile.update({

                'min': float(non_null.min()) if len(non_null) else None,

                'p50': float(non_null.median()) if len(non_null) else None,

                'max': float(non_null.max()) if len(non_null) else None,

                'mean': float(non_null.mean()) if len(non_null) else None,

                'std': float(non_null.std(ddof=0)) if len(non_null) else None,

            })

        elif pd.api.types.is_datetime64_any_dtype(s):

            profile.update({

                'min': str(non_null.min()) if len(non_null) else None,

                'max': str(non_null.max()) if len(non_null) else None,

            })

        else:

            top_vals = non_null.astype(str).value_counts().head(5)

            profile.update({

                'top_values': {k: int(v) for k, v in top_vals.to_dict().items()}

            })



        col_profiles.append(profile)



    return {

        'dataset_name': name,

        'shape': {'rows': int(df.shape[0]), 'cols': int(df.shape[1])},

        'columns': col_profiles

    }



merged_profile = profile_dataframe(merged, 'olist_processed')

daily_profile = profile_dataframe(daily.reset_index().rename(columns={'index': 'date'}), 'daily_features')



with open(RAG_DIR / 'dataset_metadata_profile.json', 'w', encoding='utf-8') as f:

    json.dump(merged_profile, f, ensure_ascii=True, indent=2)



with open(RAG_DIR / 'daily_metadata_profile.json', 'w', encoding='utf-8') as f:

    json.dump(daily_profile, f, ensure_ascii=True, indent=2)



# Per-table raw metadata profiles for the 8 source tables

raw_tables = {

    'orders': orders,

    'order_items': order_items,

    'payments': payments,

    'reviews': reviews,

    'customers': customers,

    'sellers': sellers,

    'products': products,

    'geolocation': geolocation,

}



raw_profiles = {}

for table_name, df_table in raw_tables.items():

    table_profile = profile_dataframe(df_table, table_name)

    raw_profiles[table_name] = table_profile

    with open(RAG_DIR / f'{table_name}_metadata_profile.json', 'w', encoding='utf-8') as f:

        json.dump(table_profile, f, ensure_ascii=True, indent=2)



print('Saved metadata profiles to:', RAG_DIR)

Saved metadata profiles to: processed\rag


In [28]:
# Create retrieval chunks and business-friendly briefing text (grounded in processed data)

def make_summary_facts(merged_df: pd.DataFrame, daily_df: pd.DataFrame):

    facts = {

        'rows': int(len(merged_df)),

        'orders': int(merged_df['order_id'].nunique()),

        'customers': int(merged_df['customer_id'].nunique()),

        'products': int(merged_df['product_id'].nunique()),

        'sellers': int(merged_df['seller_id'].nunique()),

        'date_min': str(merged_df['order_purchase_date'].min()),

        'date_max': str(merged_df['order_purchase_date'].max()),

        'late_delivery_rate': float(merged_df['is_late_delivery'].mean()),

        'avg_review_score': float(merged_df['review_score'].mean()),

        'total_revenue': float(merged_df['payment_value_raw'].sum()),

        'median_ticket': float(merged_df['payment_value_raw'].median()),

    }

    facts['top_payment_types'] = (

        merged_df['payment_type'].value_counts(normalize=True).head(5).round(4).to_dict()

    )

    facts['top_categories'] = (

        merged_df['product_category_name'].value_counts().head(10).to_dict()

    )

    facts['daily_corr'] = (

        daily_df[['total_orders', 'total_revenue', 'avg_review_score', 'late_delivery_rate']]

        .corr().round(3).to_dict()

    )

    return facts



facts = make_summary_facts(merged, daily)



business_brief = f"""

Business Data Brief (Grounded in Processed Olist Data)



Coverage

- Time period: {facts['date_min']} to {facts['date_max']}

- Orders: {facts['orders']:,}

- Customers: {facts['customers']:,}

- Sellers: {facts['sellers']:,}



Commercial Health

- Total modeled revenue: {facts['total_revenue']:.2f}

- Median order ticket: {facts['median_ticket']:.2f}

- Late delivery rate: {facts['late_delivery_rate']:.2%}

- Average review score: {facts['avg_review_score']:.2f}



What this means

- The data supports both demand-side analysis (orders/revenue trends) and service-quality analysis (delivery and reviews).

- Payment mix and category concentration can explain revenue swings and campaign effectiveness.

- Daily features are ready for market-context joins, helping tie business operations to external market signals.

""".strip()



chunks = []

chunks.append({

    'chunk_id': 'olist_overview_001',

    'source': 'olist_processed',

    'chunk_type': 'dataset_summary',

    'text': business_brief

})



for col_profile in merged_profile['columns']:

    txt = (

        f"Column {col_profile['column']} ({col_profile['dtype']}) in {col_profile['dataset']}: "

        f"null_rate={col_profile['null_rate']:.4f}, n_unique={col_profile['n_unique']}, "

        f"sample_values={col_profile['sample_values']}"

    )

    chunks.append({

        'chunk_id': f"schema_{col_profile['column']}",

        'source': 'olist_processed',

        'chunk_type': 'schema_stats',

        'text': txt

    })



# Add one summary chunk per raw source table profile

for table_name, table_profile in raw_profiles.items():

    col_count = table_profile['shape']['cols']

    row_count = table_profile['shape']['rows']

    top_null_cols = sorted(

        table_profile['columns'],

        key=lambda x: x['null_rate'],

        reverse=True,

    )[:5]

    null_desc = ', '.join([

        f"{c['column']} ({c['null_rate']:.2%} null)" for c in top_null_cols

    ])

    table_summary_text = (

        f"Raw table {table_name}: rows={row_count}, cols={col_count}. "

        f"Highest-null columns: {null_desc}."

    )

    chunks.append({

        'chunk_id': f'raw_table_{table_name}_summary',

        'source': table_name,

        'chunk_type': 'raw_table_schema',

        'text': table_summary_text,

    })



finnhub_join_note = {

    'chunk_id': 'finnhub_join_note_001',

    'source': 'join_strategy',

    'chunk_type': 'integration_guidance',

    'text': (

        'Join daily_features with Finnhub OHLC or sector features on UTC date. '

        'Use lagged market features (t-1) when predicting business outcomes to avoid leakage. '

        'When generating business insights, explain impact in plain language: revenue, demand, fulfillment risk, and customer experience.'

    )

}

chunks.append(finnhub_join_note)



with open(RAG_DIR / 'context_chunks.jsonl', 'w', encoding='utf-8') as f:

    for rec in chunks:

        f.write(json.dumps(rec, ensure_ascii=True) + '\n')



with open(RAG_DIR / 'business_brief.txt', 'w', encoding='utf-8') as f:

    f.write(business_brief + '\n')



print('Saved chunk file:', RAG_DIR / 'context_chunks.jsonl')

print('Saved business brief:', RAG_DIR / 'business_brief.txt')

print('Chunk count:', len(chunks))

Saved chunk file: processed\rag\context_chunks.jsonl
Saved business brief: processed\rag\business_brief.txt
Chunk count: 102


In [29]:
import requests



FINNHUB_API_KEY = "d6tsnn1r01qhkb45fil0d6tsnn1r01qhkb45filg"



def fetch_finnhub_daily(symbol: str, from_date: str, to_date: str) -> pd.DataFrame:

    """

    Fetch daily OHLCV candles from Finnhub for a given symbol.

    from_date and to_date: 'YYYY-MM-DD' strings matching Olist date range.

    Returns a DataFrame with columns: date, open, high, low, close, volume

    aligned to UTC daily index for joining with daily_features.csv

    """

    import time

    url = "https://finnhub.io/api/v1/stock/candle"

    from_ts = int(pd.Timestamp(from_date, tz='UTC').timestamp())

    to_ts   = int(pd.Timestamp(to_date,   tz='UTC').timestamp())

    resp = requests.get(url, params={

        "symbol": symbol, "resolution": "D",

        "from": from_ts, "to": to_ts, "token": FINNHUB_API_KEY

    })

    data = resp.json()

    if data.get("s") != "ok":

        raise ValueError(f"Finnhub returned status: {data.get('s')}")

    df = pd.DataFrame({

        "date":   pd.to_datetime(data["t"], unit="s", utc=True).floor("D"),

        "open":   data["o"], "high": data["h"],

        "low":    data["l"], "close": data["c"], "volume": data["v"]

    }).set_index("date")

    return df



# Example: fetch Brazilian market index proxy for Olist date range

# finnhub_df = fetch_finnhub_daily("EWZ", "2016-09-04", "2018-10-17")

# combined = daily.join(finnhub_df, how='left')

# combined.to_csv(OUTPUT_DIR / "daily_features_with_market.csv")

# Add a Finnhub metadata chunk to context_chunks.jsonl after fetching.